# Stage 3 — DPO Preference Alignment (Healthcare FAQ Assistant)

**Goal:** use Direct Preference Optimization to make the SFT model *prefer* answers that are correct, helpful, safe, professional and domain-specific over weak/unsafe/generic ones (`data/preference_dataset.jsonl`, 50+ triples).

Pipeline: Base → Stage 1 → Stage 2: SFT → **[Stage 3: DPO]** → Final Assistant

> ⚠️ Educational project — general health information only, not medical advice.

## 0. Install dependencies (Colab)

In [1]:
# Run once on a fresh Colab GPU runtime (Runtime -> Change runtime type -> T4 GPU).
# Unsloth installs compatible transformers / peft / trl / bitsandbytes itself --
# do NOT pin an old trl (e.g. trl<0.12): it passes `tokenizer=` to Trainer.__init__(),
# which newer transformers removed, causing:
#   TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'
%%capture
!pip install -q unsloth unsloth_zoo
# After installing, restart the runtime once (Runtime -> Restart session) before continuing.


In [2]:
import unsloth  # Important: import Unsloth early
import torch
import time
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
assert torch.cuda.is_available(), "No GPU detected. In Colab: Runtime -> Change runtime type -> T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: Tesla T4


## 0b. Colab bootstrap — get repo files & set REPO_DIR

In [3]:
# ============================================================
#  COLAB BOOTSTRAP  --  make repo files available + set REPO_DIR
#  Pick ONE method by setting BOOTSTRAP below.
# ============================================================
import os

BOOTSTRAP = "drive"   # "drive" (recommended) | "clone" | "local"

if BOOTSTRAP == "drive":
    # 1) Copy the `healthcare-ai-assistant-finetuning` folder into your Google Drive.
    # 2) Adjust the path below if you placed it somewhere other than MyDrive root.
    #    Drive is recommended because outputs/ persist across sessions, so Stage 1->2->3 chain.
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["REPO_DIR"] = "/content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning"

elif BOOTSTRAP == "clone":
    # Ephemeral: /content is wiped on disconnect. Run all stages in one session,
    # or set PUSH_TO_HUB=True so each stage's model is saved to the Hugging Face Hub.
    REPO_URL = "https://github.com/your-username/healthcare-faq-assistant.git"
    DEST = "/content/healthcare-faq-assistant"
    if not os.path.isdir(DEST):
        os.system(f"git clone {REPO_URL} {DEST}")
    os.environ["REPO_DIR"] = DEST

else:  # "local" -- running outside Colab, from inside the notebooks/ folder
    os.environ["REPO_DIR"] = ".."

print("REPO_DIR =", os.environ.get("REPO_DIR"))
assert os.path.isdir(os.path.join(os.environ["REPO_DIR"], "data")), \
    "REPO_DIR is wrong: no data/ folder found. Fix the path in this cell."

Mounted at /content/drive
REPO_DIR = /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning


## 1. Select base model (must match the model used for SFT)

In [4]:
# ============================================================
#  MODEL SELECTION  --  change MODEL_NAME to switch base model
# ============================================================
MODEL_OPTIONS = {
    "qwen2.5-0.5b":   "unsloth/Qwen2.5-0.5B",
    "llama-3.2-1b":   "unsloth/Llama-3.2-1B",
    "qwen2.5-1.5b":   "unsloth/Qwen2.5-1.5B",
    "tinyllama-1.1b": "unsloth/tinyllama",
    "gemma-2-2b":     "unsloth/gemma-2-2b",
}

MODEL_NAME = "qwen2.5-0.5b"   # <-- change this one line to pick a model
MODEL_REPO = MODEL_OPTIONS[MODEL_NAME]
print(f"Selected model: {MODEL_NAME}  ->  {MODEL_REPO}")

Selected model: qwen2.5-0.5b  ->  unsloth/Qwen2.5-0.5B


## 2. Paths & system prompt

In [5]:
import os

# If you cloned the repo in Colab, point REPO_DIR at the repo root.
# This notebook lives in <repo>/notebooks/, so the repo root is one level up.
REPO_DIR   = os.environ.get("REPO_DIR", "..")
DATA_DIR   = os.path.join(REPO_DIR, "data")
OUTPUT_DIR = os.path.join(REPO_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Data dir:  ", os.path.abspath(DATA_DIR))
print("Output dir:", os.path.abspath(OUTPUT_DIR))

Data dir:   /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/data
Output dir: /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs


In [6]:
# Shared system prompt used for instruction formatting / inference
SYSTEM_PROMPT = (
    "You are a Healthcare FAQ Assistant. You provide clear, general health information for "
    "educational purposes only. You are not a substitute for professional medical advice, "
    "diagnosis, or treatment. Always recommend consulting a qualified healthcare professional, "
    "and advise seeking emergency care for urgent symptoms."
)

## 2b. Hugging Face Hub config (optional)

In [ ]:
# ============================================================
#  HUGGING FACE HUB  --  set these to push your trained models
# ============================================================
PUSH_TO_HUB  = False                 # set True to upload after training
HF_USERNAME  = "your-hf-username"     # <-- your Hugging Face username
HF_TOKEN     = ""                     # <-- a WRITE token from https://huggingface.co/settings/tokens

# In Colab you can store the token as a secret instead of pasting it:
#   from google.colab import userdata
#   HF_TOKEN = userdata.get('HF_TOKEN')

if PUSH_TO_HUB and HF_TOKEN:
    from huggingface_hub import login
    login(HF_TOKEN)
    print("Logged in to Hugging Face Hub as", HF_USERNAME)
else:
    print("PUSH_TO_HUB disabled (or no token). Models will be saved locally only.")

## 3. Load the SFT model

We load `outputs/stage2_merged` (the Stage-2 SFT model). If it is missing we fall back to the base model so the notebook still runs end-to-end.

In [7]:
from unsloth import FastLanguageModel, PatchDPOTrainer
PatchDPOTrainer()   # must be called before building the DPOTrainer
import torch, os

max_seq_length = 2048
STAGE2_MERGED = os.path.join(OUTPUT_DIR, "stage2_merged")
load_from = STAGE2_MERGED if os.path.isdir(STAGE2_MERGED) else MODEL_REPO
print("Loading SFT model from:", load_from)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = load_from,
    max_seq_length = max_seq_length,
    dtype          = None,
    load_in_4bit   = True,
)

Loading SFT model from: /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage2_merged
==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The tokenizer you are loading from '/content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage2_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage2_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


## 4. Set the chat template

In [8]:
from unsloth.chat_templates import get_chat_template
if tokenizer.chat_template is None:
    tokenizer = get_chat_template(tokenizer, chat_template="chatml")
    print("Applied fallback ChatML template.")
else:
    print("Using the models built-in chat template.")

Using the models built-in chat template.


## 5. Attach LoRA adapters for DPO

In [9]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth 2026.6.9 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


## 6. Load & format the preference dataset

DPO needs three fields per row: **prompt**, **chosen**, **rejected**. The prompt is rendered with the chat template (ending with the assistant generation prompt); chosen/rejected are the answer texts.

In [10]:
import json, os
from datasets import Dataset

path = os.path.join(DATA_DIR, "preference_dataset.jsonl")
rows = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
print(f"Loaded {len(rows)} preference examples")

EOS = tokenizer.eos_token
def to_dpo(ex):
    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": ex["prompt"]},
        ],
        tokenize=False, add_generation_prompt=True,
    )
    return {
        "prompt":   prompt,
        "chosen":   ex["chosen"]   + EOS,
        "rejected": ex["rejected"] + EOS,
    }

dpo_dataset = Dataset.from_list([to_dpo(r) for r in rows])
print("\nExample prompt:\n", dpo_dataset[0]["prompt"][:400])
print("\nChosen:  ", dpo_dataset[0]["chosen"][:120])
print("Rejected:", dpo_dataset[0]["rejected"][:120])

Loaded 56 preference examples

Example prompt:
 <|im_start|>system
You are a Healthcare FAQ Assistant. You provide clear, general health information for educational purposes only. You are not a substitute for professional medical advice, diagnosis, or treatment. Always recommend consulting a qualified healthcare professional, and advise seeking emergency care for urgent symptoms.<|im_end|>
<|im_start|>user
How can I apply for sick leave when I 

Chosen:   The flu often comes on suddenly with high fever, body aches, and fatigue, so rest and recovery are important. Follow you
Rejected: Just take some antibiotics and go back to work, you'll be fine.<|im_end|>


## 7. Configure and run DPO training

In [11]:
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,                  # Unsloth/PEFT uses the frozen base as the implicit reference
    processing_class = tokenizer,      # new API: was `tokenizer=` in older trl
    train_dataset = dpo_dataset,
    args = DPOConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,
        num_train_epochs = 1,
        learning_rate = 5e-6,         # DPO uses a much smaller LR than SFT
        beta = 0.1,                   # DPO temperature
        max_length = 1024,
        max_prompt_length = 512,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.0,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = os.path.join(OUTPUT_DIR, "stage3_logs"),
        report_to = "none",
    ),
)
dpo_stats = dpo_trainer.train()
dpo_stats


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Extracting prompt in train dataset (num_proc=6):   0%|          | 0/56 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=6):   0%|          | 0/56 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=6):   0%|          | 0/56 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 56 | Num Epochs = 1 | Total steps = 7
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
1,0.693147,0.000000,0.000000,0.000000,0.000000,-88.811218,-81.639099,-1.205533,-0.398639
2,0.693147,0.000000,0.000000,0.000000,0.000000,-90.137054,-81.916245,-0.996886,-0.252538
3,0.684021,0.008397,-0.009951,1.000000,0.018349,-101.523178,-78.411011,-0.956315,-0.400900
4,0.675552,0.025849,-0.009685,1.000000,0.035535,-113.151123,-84.402161,-0.975077,-0.370862
5,0.657133,0.043452,-0.029975,1.000000,0.073427,-114.161514,-84.957611,-0.866738,-0.601066
6,0.648629,0.057321,-0.033873,1.000000,0.091194,-100.275726,-75.599777,-1.024542,-0.496982
7,0.638913,0.062457,-0.049285,1.000000,0.111742,-94.975777,-83.822792,-1.185593,-0.501675


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage3_logs/checkpoint-7/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage3_logs/checkpoint-7.


TrainOutput(global_step=7, training_loss=0.6700773664883205, metrics={'train_runtime': 30.7121, 'train_samples_per_second': 1.823, 'train_steps_per_second': 0.228, 'total_flos': 0.0, 'train_loss': 0.6700773664883205, 'epoch': 1.0})

## 8. Save the DPO-aligned (final) model

In [12]:
STAGE3_ADAPTER = os.path.join(OUTPUT_DIR, "stage3_dpo")
STAGE3_MERGED  = os.path.join(OUTPUT_DIR, "stage3_merged")

model.save_pretrained(STAGE3_ADAPTER)
tokenizer.save_pretrained(STAGE3_ADAPTER)
print("Saved DPO adapter ->", STAGE3_ADAPTER)

model.save_pretrained_merged(STAGE3_MERGED, tokenizer, save_method="merged_16bit")
print("Saved final merged model ->", STAGE3_MERGED)

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage3_dpo/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage3_dpo.


Saved DPO adapter -> /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage3_dpo
Detected local model directory: /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage2_merged
Copied tokenizer.model from local model directory


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage3_merged/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:27<00:00, 27.93s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage3_merged`
Saved final merged model -> /content/drive/MyDrive/Colab Notebooks/healthcare-ai-assistant-finetuning/outputs/stage3_merged


### Optional — push the Stage model to the Hugging Face Hub

Runs only if `PUSH_TO_HUB = True` and a write token is set above. Pushes both the LoRA adapter (small) and the merged 16-bit model.

In [ ]:
if PUSH_TO_HUB and HF_TOKEN:
    repo_adapter = f"{HF_USERNAME}/healthcare-faq-{MODEL_NAME}-stage3-dpo"
    repo_merged  = f"{HF_USERNAME}/healthcare-faq-{MODEL_NAME}-stage3-dpo-merged"

    # LoRA adapter
    model.push_to_hub(repo_adapter, token=HF_TOKEN)
    tokenizer.push_to_hub(repo_adapter, token=HF_TOKEN)
    print("Pushed adapter ->", repo_adapter)

    # Merged 16-bit model (ready for inference / vLLM / TGI)
    model.push_to_hub_merged(repo_merged, tokenizer, save_method="merged_16bit", token=HF_TOKEN)
    print("Pushed merged model ->", repo_merged)
else:
    print("Skipped Hub upload (PUSH_TO_HUB is False or no token).")

### Reload this model later from the Hugging Face Hub

Once pushed, you (or anyone) can load it directly — no retraining needed:

```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "your-hf-username/healthcare-faq-qwen2.5-0.5b-stage3-dpo-merged",
    max_seq_length = 2048, load_in_4bit = True,
)
```

Or with plain Transformers (CPU or GPU):

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
repo = "your-hf-username/healthcare-faq-qwen2.5-0.5b-stage3-dpo-merged"
tokenizer = AutoTokenizer.from_pretrained(repo)
model = AutoModelForCausalLM.from_pretrained(repo)
```

## 9. Test the model after DPO

In [13]:
FastLanguageModel.for_inference(model)

def ask(question, max_new_tokens=200):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens, use_cache=True,
                         do_sample=False, temperature=0.0)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

for q in [
    "How can I apply for sick leave when I have the flu?",
    "What temperature is considered a fever in adults?",
    "Should I take antibiotics for a common cold?",
    "Can I stop my blood pressure medication if I feel fine?",
    "What should I do if my 2-month-old baby has a fever?",
    "How do I recognize the signs of a stroke?",
    "How much paracetamol can I take if my headache won't go away?",
    "What's a good way to lose weight quickly?",
    "How can I manage my type 2 diabetes?",
    "What can this assistant help me with?",
]:
    print("Q:", q)
    print("A:", ask(q))
    print("-" * 80)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How can I apply for sick leave when I have the flu?


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

A: You are not legally allowed to take sick leave for the flu. You are expected to work the full sick leave period, which is usually 10 days. You can apply for sick leave during the flu season, but you are not allowed to take it during the flu season. You are expected to work the full sick leave period, which is usually 10 days. You can apply for sick leave during the flu season, but you are not legally allowed to take it during the flu season. You are expected to work the full sick leave period, which is usually 10 days. You can apply for sick leave during the flu season, but you are not legally allowed to take it during the flu season. You are expected to work the full sick leave period, which is usually 10 days. You can apply for sick leave during the flu season, but you are not legally allowed to take it during the flu season. You are expected to work the full sick leave period, which
--------------------------------------------------------------------------------
Q: What temperatu

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

A: Most adults can take paracetamol for a headache without a problem. However, if you have a headache that is very severe, is very frequent, comes with a fever, or is very painful, you should tell your clinician right away. You may need paracetamol to help your headache. You should not take it for a headache that is not caused by a fever or a headache that is very severe, because it can cause serious side effects. You should always recommend consulting a clinician, who can advise whether paracetamol is right for you.
--------------------------------------------------------------------------------
Q: What's a good way to lose weight quickly?


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Start with a diet that is lower in calories and higher in fiber, and include plenty of fruits, vegetables, whole grains, and healthy fats. You can also include foods that are high in fiber, such as fruits, vegetables, whole grains, and legumes, and foods that are low in fat, such as lean meats, fish, and beans. A dietitian can help you plan a meal plan that meets your needs and goals. You can also use apps that help you track your food and calories. A clinician can advise on a diet that is right for you.
--------------------------------------------------------------------------------
Q: How can I manage my type 2 diabetes?


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Eat a balanced diet, stay active, and take your medication as directed. Your clinician can advise on the best diet for you and advise on how often you need insulin. You are your own clinician, so you are the best person to advise you on your diabetes. You are a guide only, and you are not a substitute for professional medical advice, diagnosis, or treatment. Always recommend consulting a qualified clinician, who can advise on your diabetes and advise on the best diet for you. You are a guide only, and you are not a substitute for professional medical advice, diagnosis, or treatment. Always recommend consulting a qualified clinician, who can advise on your diabetes and advise on the best diet for you. You are a guide only, and you are not a substitute for professional medical advice, diagnosis, or treatment. Always recommend consulting a qualified clinician, who can advise on your diabetes and advise on the best diet for you. You are a guide only, and you are not
--------------------

## Done — Stage 3 complete ✅ (Final Healthcare FAQ Assistant)

The final model is in `outputs/stage3_merged`. Use it from `src/inference.py`, and fill in the comparison reports (`reports/`) using the base, SFT and DPO answers.